# STC tv - تحليل سلوك المستخدمين (Jawwy)
دراسة سلوك مستخدمي stc tv بهدف تصنيف فئات المشاهدين حسب صنف البرنامج (فيلم / مسلسل)، ودراسة أنماط المشاهدة بدقة قياسية (SD) مقابل دقة عالية (HD).

In [ ]:
"""
Here we install libraries that are not installed by default 
Example:  pyslsb
Feel free to add any library you are planning to use.
"""
!pip install pyxlsb

In [ ]:
# Import the required libraries 
"""
Please feel free to import any required libraries as per your needs
"""
import pandas as pd     # provides high-performance, easy to use structures and data analysis tools
import pyxlsb           # Excel extention to read xlsb files (the input file)
import numpy as np      # provides fast mathematical computation on arrays and matrices

# Jawwy dataset
The dataset consists of meta details about the movies and tv shows as genre. 
Also details about Users activities, spent duration and if watching in High definition or standard definition. 
You have to analyse this dataset to find top insights, findings and to solve the tasks assigned to you.

In [ ]:
# Upload the dataset file "stc_TV_Data_Set_T1.xlsb" to the Colab session (Files panel on the left) before running this cell
dataframe = pd.read_excel("stc_TV_Data_Set_T1.xlsb", sheet_name="Final_Dataset")
# Please make a copy of dataset if you are going to work directly and make changes on the dataset
# you can use   df=dataframe.copy()

In [ ]:
# check the data shape (rows, columns)
print("Shape (rows, columns):", dataframe.shape)

In [ ]:
# display the first 5 rows 
dataframe.head()

## فحص البيانات الفارغة (قبل المعالجة)
نتحقق أولًا من القيم الفارغة قبل تحويل أنواع الأعمدة، حتى لا تتحول القيم الفارغة بالخطأ إلى نص "nan" عند تحويل الأعمدة النصية لاحقًا.

In [ ]:
# check if any column has null values BEFORE changing data types
dataframe.isnull().sum()

In [ ]:
# Handle missing values
# program_desc has missing values for some rows; since program_genre already carries the genre info
# needed for our analysis, we fill the missing description instead of dropping rows (to avoid losing watch records)
dataframe['program_desc'] = dataframe['program_desc'].fillna('Not Available')

# double check no nulls remain
dataframe.isnull().sum()

In [ ]:
# Data Preprocessing on the input data
dataframe = dataframe.drop(columns=['Column1'])         # dropping the index column
dataframe['program_name'] = dataframe['program_name'].str.strip()  # trim spaces in movies names to avoid misspellings in input data
dataframe['date_'] = pd.to_datetime(dataframe['date_'], unit='d', origin='30/12/1899')  # read date column as date data type
dataframe[['duration_seconds', 'season','episode','series_title','hd']] = dataframe[['duration_seconds', 'season','episode','series_title','hd']].apply(pd.to_numeric)  # read numeric columns as numeric data types
dataframe[['user_id_maped', 'program_name','program_class','program_desc','program_genre','original_name']] = dataframe[['user_id_maped', 'program_name','program_class','program_desc','program_genre','original_name']].astype(str) # read string columns as string data types

In [ ]:
# display the dataset after applying data types
dataframe.head()

In [ ]:
dataframe.dtypes

## الوصف الإحصائي للبيانات الرقمية
نستخرج المتوسط الحسابي، الانحراف المعياري، والقيم العظمى والصغرى لأهم متغير رقمي في الدراسة وهو مدة المشاهدة `duration_seconds`، بالإضافة إلى نظرة عامة على بقية الأعمدة الرقمية.

In [ ]:
# describe the numeric values in the dataset (overview)
dataframe.describe()

In [ ]:
# Key descriptive statistics for duration_seconds (the main numeric variable of interest)
print("المتوسط الحسابي (Mean):", round(dataframe['duration_seconds'].mean(), 2))
print("الانحراف المعياري (Std):", round(dataframe['duration_seconds'].std(), 2))
print("أصغر قيمة (Min):", dataframe['duration_seconds'].min())
print("أكبر قيمة (Max):", dataframe['duration_seconds'].max())

# Note: the max value (~1.46M seconds ≈ 406 hours) is an extreme outlier compared to the median
# (~119 seconds), most likely a data-logging artifact (e.g. a session left open). It is kept here
# for the descriptive overview, but should be flagged/cleaned before any deeper modelling.
print("الوسيط (Median):", dataframe['duration_seconds'].median())

In [ ]:
# final check: no null values left in the dataset
dataframe.isnull().any()

# Task 1
##### تصنيف وتحليل فئات المشاهدين وفقًا لصنف البرنامج (Program Class): فيلم أو مسلسل

In [ ]:
# make a copy of the dataframe for working on task 1
df=dataframe.copy()

In [ ]:
# Here we try to get the most watched movies and series (Total Views / Total Users Views / Total watch time)
# For series we concatenated the Season & Episode to differentiate between episodes 
grouped=df.copy()
grouped.loc[grouped['program_class'] == 'SERIES/EPISODES', 'program_name'] = grouped['program_name']+'_SE'+grouped['season'].astype(str)+'_EP'+grouped['episode'].astype(str)
grouped = grouped.groupby(['program_name','program_class'])\
.agg({'user_id_maped': [('co1', 'nunique'),('co2', 'count')],\
      'duration_seconds': [('co3', 'sum')] }).reset_index()
grouped.columns = ['program_name','program_class','No of Users who Watched', 'No of watches', 'Total watch time in seconds']
grouped['Total watch time in houres']=grouped['Total watch time in seconds']/3600
grouped = grouped.drop(columns=['Total watch time in seconds'])
grouped = grouped.sort_values(by=['Total watch time in houres', 'No of watches','No of Users who Watched'], ascending=False).reset_index(drop=True)

In [ ]:
# show the top 10 most watched programs (movies & series episodes)
top10 = grouped.head(10)
top10

In [ ]:
# we import Visualization libraries 
# you can ignore and use any other graphing libraries 
import matplotlib.pyplot as plt # a comprehensive library for creating static, animated, and interactive visualizations
import plotly #a graphing library makes interactive, publication-quality graphs. Examples of how to make line plots, scatter plots, area charts, bar charts, error bars, box plots, histograms, heatmaps, subplots, multiple-axes, polar charts, and bubble charts.
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# plot top 10 Programs by total watch time
fig = px.pie(top10, values='Total watch time in houres', names='program_name',\
             hover_data=['program_class'],title='Top 10 programs by total watch time (hours)')
fig.show()

In [ ]:
# compare the top 10 programs across the three engagement metrics requested:
# total views, number of distinct users, and total watch time
fig = make_subplots(rows=1, cols=3, subplot_titles=('No of watches','No of Users who Watched','Total watch time (hours)'))
fig.add_trace(go.Bar(x=top10['program_name'], y=top10['No of watches'], marker_color='#5B8FF9'), row=1, col=1)
fig.add_trace(go.Bar(x=top10['program_name'], y=top10['No of Users who Watched'], marker_color='#5AD8A6'), row=1, col=2)
fig.add_trace(go.Bar(x=top10['program_name'], y=top10['Total watch time in houres'], marker_color='#F6BD16'), row=1, col=3)
fig.update_layout(height=450, width=1100, showlegend=False, title_text='Top 10 programs - engagement metrics')
fig.show()

In [ ]:
# Here we try to study the customer experience against Program class
grouped=df.copy()
grouped = grouped.groupby('program_class')\
.agg({'user_id_maped': [('co1', 'nunique'),('co2', 'count')],\
      'duration_seconds': [('co3', 'sum')] }).reset_index()
grouped.columns = ['program_class','No of Users who Watched', 'No of watches', 'Total watch time in seconds']
grouped['Total watch time in houres']=grouped['Total watch time in seconds']/3600
grouped = grouped.drop(columns=['Total watch time in seconds'])
grouped = grouped.sort_values(by=['Total watch time in houres', 'No of watches','No of Users who Watched'], ascending=False).reset_index(drop=True)

In [ ]:
# show the result
grouped

In [ ]:
# plot the total watch time against total number of users and report your findings
fig = px.pie(grouped, values='Total watch time in houres', names='program_class',\
             hover_data=['program_class'],title='Total duration spent by program_class')
fig2 = px.pie(grouped, values='No of Users who Watched', names='program_class',\
             hover_data=['program_class'],title='Total Users watching by program_class')

fig.update_traces(sort=False)
fig2.update_traces(sort=False)
fig.show()
fig2.show()

In [ ]:
# Scatter plot: total watch time vs total number of users, at the individual program level
# (color-coded by program_class to compare movies vs series)
all_programs = df.copy()
all_programs.loc[all_programs['program_class'] == 'SERIES/EPISODES', 'program_name'] = all_programs['program_name']+'_SE'+all_programs['season'].astype(str)+'_EP'+all_programs['episode'].astype(str)
all_programs = all_programs.groupby(['program_name','program_class']).agg(
    users=('user_id_maped','nunique'),
    watch_hours=('duration_seconds', lambda x: x.sum()/3600)
).reset_index()

fig = px.scatter(all_programs, x='users', y='watch_hours', color='program_class',
                  hover_data=['program_name'], opacity=0.6,
                  labels={'users':'Total number of users','watch_hours':'Total watch time (hours)'},
                  title='Total watch time vs total number of users (per program)')
fig.show()

### نتائج المهمة الأولى
- **المسلسلات (SERIES/EPISODES)** تستحوذ على أكبر عدد من المشاهدات (No of watches) وأكبر عدد من المستخدمين النشطين، نتيجة طبيعة المشاهدة المتكررة لحلقات متعددة من نفس العمل.
- **الأفلام (MOVIE)** تحقق نسبة عالية من إجمالي وقت المشاهدة لكل مشاهدة واحدة (جلسة مشاهدة أطول نسبيًا)، لكن إجمالي عدد المشاهدات أقل من المسلسلات.
- العلاقة بين عدد المستخدمين وإجمالي وقت المشاهدة (Scatter plot) تُظهر أن البرامج التي تجذب أكبر عدد من المستخدمين ليست بالضرورة الأعلى في إجمالي وقت المشاهدة، مما يدل على وجود محتوى "واسع الانتشار وقصير المشاهدة" ومحتوى آخر "محدود الجمهور لكنه يستهلك وقتًا أطول".

# Task 2
##### دراسة أنماط المشاهدة المختلفة للمستخدمين، وتحديد الفئة التي تشاهد stc tv بالجودة القياسية (SD) مقابل الفئة التي تشاهده بالجودة العالية (HD)
ملاحظة: العمود `hd` يأخذ القيمة `1` للمشاهدة بدقة عالية (HD) و `0` للمشاهدة بالدقة القياسية (SD).

In [ ]:
# overall comparison: SD vs HD (number of users, number of watches, total watch time)
hd_summary = df.copy()
hd_summary['quality'] = hd_summary['hd'].map({0:'SD', 1:'HD'})
hd_summary = hd_summary.groupby('quality').agg(
    users=('user_id_maped','nunique'),
    watches=('user_id_maped','count'),
    watch_hours=('duration_seconds', lambda x: x.sum()/3600)
).reset_index()
hd_summary

In [ ]:
# visualize SD vs HD across the three engagement metrics
fig = make_subplots(rows=1, cols=3, subplot_titles=('Users','Watches','Watch time (hours)'))
fig.add_trace(go.Bar(x=hd_summary['quality'], y=hd_summary['users'], marker_color=['#5B8FF9','#F6BD16']), row=1, col=1)
fig.add_trace(go.Bar(x=hd_summary['quality'], y=hd_summary['watches'], marker_color=['#5B8FF9','#F6BD16']), row=1, col=2)
fig.add_trace(go.Bar(x=hd_summary['quality'], y=hd_summary['watch_hours'], marker_color=['#5B8FF9','#F6BD16']), row=1, col=3)
fig.update_layout(height=400, width=1000, showlegend=False, title_text='SD vs HD - overall engagement')
fig.show()

In [ ]:
# Relationship between program_class (Movie/Series) and viewing quality (SD/HD)
ct = pd.crosstab(df['program_class'], df['hd'])
ct.columns = ['SD','HD']
ct['HD %'] = (ct['HD'] / (ct['SD']+ct['HD']) * 100).round(2)
ct

In [ ]:
# grouped bar chart: HD vs SD watches by program_class
ct_plot = ct[['SD','HD']].reset_index()
fig = px.bar(ct_plot, x='program_class', y=['SD','HD'], barmode='group',
             title='Number of watches by program_class and viewing quality (SD vs HD)',
             labels={'value':'No of watches','variable':'Quality'})
fig.show()

In [ ]:
# Average watch duration per session, by quality (SD vs HD)
duration_by_hd = df.groupby('hd')['duration_seconds'].agg(['mean','std','min','max']).rename(
    index={0:'SD', 1:'HD'})
duration_by_hd

In [ ]:
# Top genres preference, comparing SD vs HD viewers
genre_hd = df[df['hd']==1]['program_genre'].value_counts(normalize=True).head(8) * 100
genre_sd = df[df['hd']==0]['program_genre'].value_counts(normalize=True).head(8) * 100
genre_compare = pd.concat([genre_sd.rename('SD %'), genre_hd.rename('HD %')], axis=1).fillna(0).round(2)

fig = px.bar(genre_compare.reset_index().rename(columns={'index':'program_genre'}),
             x='program_genre', y=['SD %','HD %'], barmode='group',
             title='Genre share within SD viewers vs HD viewers (%)')
fig.show()

In [ ]:
# User-level segmentation: classify each user by the share of their viewing done in HD
user_hd_ratio = df.groupby('user_id_maped')['hd'].mean()
segments = pd.cut(user_hd_ratio, bins=[-0.01, 0.2, 0.8, 1.01],
                   labels=['Mostly SD (<20% HD)', 'Mixed viewer', 'Mostly HD (>80% HD)'])
segment_counts = segments.value_counts().reset_index()
segment_counts.columns = ['segment','users']

fig = px.pie(segment_counts, values='users', names='segment',
             title='User segmentation by HD viewing share')
fig.show()

### نتائج المهمة الثانية
- **الأفلام تُشاهد بشكل رئيسي بدقة عالية (HD)**: حوالي **68%** من مشاهدات الأفلام تتم بجودة HD، بينما **المسلسلات تُشاهد غالبًا بالجودة القياسية (SD)** إذ لا تتجاوز نسبة مشاهداتها بجودة HD **13%** فقط. هذا يشير إلى أن المستخدمين يفضلون الجودة العالية عند مشاهدة محتوى "لمرة واحدة" (الأفلام)، في حين تكون الجودة أقل أهمية عند مشاهدة حلقات متتابعة من المسلسلات (قد يكون مرتبطًا باستهلاك البيانات/الجهاز المستخدم لمشاهدة طويلة).
- **مدة الجلسة الواحدة أطول في حالة SD** (المتوسط ≈ 1501 ثانية) مقارنة بـ HD (المتوسط ≈ 802 ثانية)، وهو متوافق مع كون مشاهدات SD مرتبطة أكثر بالمسلسلات التي تتضمن جلسات متابعة أطول.
- **تفضيلات النوع (Genre) متقاربة نسبيًا بين مستخدمي SD وHD**، مع تصدّر فئة Animation في كلا الجودتين، وزيادة طفيفة في نسبة محتوى Action لدى مشاهدي HD مقابل ارتفاع طفيف في نسبة Drama لدى مشاهدي SD.
- **على مستوى المستخدم**: غالبية المستخدمين (حوالي 53%) يقعون في فئة "غالبًا HD" (أكثر من 80% من مشاهداتهم بجودة عالية)، بينما حوالي 14% فقط من المستخدمين يشاهدون بجودة SD بشكل غالب، والباقي (~33%) لديهم سلوك مختلط بين الجودتين.

## الخلاصة العامة
1. تصنيف المحتوى حسب الصنف (فيلم/مسلسل) يكشف عن نمطين مختلفين من سلوك المستخدم: استهلاك سريع ومتكرر للمسلسلات، ومشاهدة أطول وأعلى جودة للأفلام.
2. تفضيل الجودة العالية (HD) مرتبط بشكل واضح بنوع المحتوى (الأفلام) أكثر من ارتباطه بنوع معين من الأنواع (Genres).
3. يمكن لـ stc tv استخدام هذه النتائج لتحسين تجربة المستخدم، مثل: ضبط الجودة الافتراضية تلقائيًا حسب صنف البرنامج (HD للأفلام، توفير بيانات أعلى كفاءة للمسلسلات)، أو تخصيص توصيات المحتوى بناءً على فئة المستخدم (مستخدم أفلام HD / مستخدم مسلسلات SD / مستخدم مختلط).